# 00 · Carga y consolidación de la cohorte

**Fase CRISP-DM: Comprensión de los datos (inicio).**

El dataset del [PhysioNet/CinC Challenge 2019](https://physionet.org/content/challenge-2019/1.0.0/)
no viene como una tabla, sino como **40.336 archivos `.psv`, uno por paciente**, repartidos en
dos carpetas que corresponden a **dos hospitales distintos** (`training_setA` y `training_setB`).

Cada archivo es una serie temporal: **una fila por hora de estancia en UCI**, con 40 columnas
(signos vitales, laboratorios, demografía) y la etiqueta `SepsisLabel`.

Este notebook tiene un único objetivo: convertir esos 40.336 archivos en **una sola tabla
confiable**, verificando por el camino que los supuestos que vamos a usar en todo el proyecto
son ciertos. No hacemos análisis todavía; hacemos **auditoría de integridad**.

### Dos piezas de información que están fuera de los datos

Antes de cargar, hay que notar algo que es fácil pasar por alto: dentro de cada `.psv` **no
existe ninguna columna que identifique al paciente ni al hospital**. Esa información vive en
el nombre del archivo (`p000001.psv`) y en la carpeta que lo contiene.

Las dos son críticas y hay que rescatarlas al cargar:

- **`pid` (paciente)**: sin él no podemos agrupar las filas por paciente, y sin eso cualquier
  partición train/test mezclaría horas del mismo paciente en ambos lados. Eso sería fuga de
  información y las métricas quedarían infladas. Es el error más grave posible en este dataset.
- **`hosp` (hospital)**: los dos sets provienen de sistemas hospitalarios distintos. Queremos
  medir si un modelo entrenado en uno funciona en el otro, y para eso necesitamos la etiqueta.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent / "src"))
import io_datos
from config import cargar_config

cfg = cargar_config()
pd.set_option("display.max_columns", 50)

## 1. Inventario de archivos

Antes de leer 40.336 archivos conviene confirmar que están todos donde esperamos y en qué
proporción, porque un desbalance entre hospitales condicionaría la estrategia de validación.

In [2]:
archivos = io_datos.listar_archivos(cfg)
inventario = pd.Series([h for _, h in archivos]).value_counts().rename("n_pacientes").to_frame()
inventario.assign(porcentaje_=lambda d: (d.n_pacientes / d.n_pacientes.sum() * 100).round(1))

,n_pacientes,porcentaje_
A,20336,50.4
B,20000,49.6


Los dos hospitales aportan prácticamente el mismo número de pacientes, así que cualquier
diferencia que encontremos después **no vendrá del tamaño de muestra** sino de la población o
de la práctica clínica de cada sitio. Es un buen punto de partida para la comparación A vs B.

## 2. Cómo es un archivo por dentro

Miramos un paciente crudo antes de tocar nada. Es la única forma de entender qué significa
una fila y de detectar el patrón de medición real.

In [3]:
ruta_ejemplo, hosp_ejemplo = archivos[0]
ejemplo = pd.read_csv(ruta_ejemplo, sep="|")
print(f"{ruta_ejemplo.name} (hospital {hosp_ejemplo}) — {ejemplo.shape[0]} horas x {ejemplo.shape[1]} columnas")
ejemplo.head(8)

p000001.psv (hospital A) — 54 horas x 41 columnas


,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,FiO2,pH,PaCO2,SaO2,AST,BUN,Alkalinephos,Calcium,Chloride,Creatinine,Bilirubin_direct,Glucose,Lactate,Magnesium,Phosphate,Potassium,Bilirubin_total,TroponinI,Hct,Hgb,PTT,WBC,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,1,0
1,97.0,95.0,NaN,98.0,75.33,NaN,19.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,2,0
2,89.0,99.0,NaN,122.0,86.00,NaN,22.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,3,0
3,90.0,95.0,NaN,NaN,NaN,NaN,30.0,NaN,24.0,NaN,NaN,7.36,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,4,0
4,103.0,88.5,NaN,122.0,91.33,NaN,24.5,NaN,NaN,NaN,0.28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,5,0
5,110.0,91.0,NaN,NaN,NaN,NaN,22.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,6,0
6,108.0,92.0,36.11,123.0,77.00,NaN,29.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,7,0
7,106.0,90.5,NaN,93.0,76.33,NaN,29.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.14,0,NaN,NaN,-0.03,8,0


Aquí ya aparece el rasgo que define este dataset: **la primera fila está casi completamente
vacía** y las siguientes solo traen algunos signos vitales. Los laboratorios aparecen de forma
esporádica, no cada hora.

Esto no es un defecto de los datos: es el reflejo de cómo funciona una UCI. Los monitores
registran frecuencia cardiaca y saturación de forma continua, pero un laboratorio se solicita
solo cuando el médico decide pedirlo. El patrón de qué se mide y cuándo sería muy interesante de analizar.

Veamos qué tan denso es el registro en este paciente concreto:

In [4]:
# Cuántas mediciones reales tiene cada variable en este paciente, sobre el total de sus horas
densidad_ejemplo = (ejemplo.notna().sum() / len(ejemplo) * 100).round(1).sort_values(ascending=False)
densidad_ejemplo.rename("% de horas con medición").to_frame().T

,SepsisLabel,Age,ICULOS,Gender,HospAdmTime,Resp,HR,O2Sat,MAP,SBP,Temp,BaseExcess,pH,PaCO2,FiO2,SaO2,Platelets,Hgb,Phosphate,Creatinine,Magnesium,HCO3,BUN,Chloride,Calcium,Glucose,Potassium,WBC,Hct,Bilirubin_total,Alkalinephos,AST,DBP,EtCO2,Bilirubin_direct,Lactate,TroponinI,Fibrinogen,PTT,Unit1,Unit2
% de horas con medición,100.0,100.0,100.0,100.0,100.0,92.6,90.7,81.5,77.8,77.8,18.5,13.0,13.0,11.1,7.4,7.4,3.7,3.7,3.7,3.7,3.7,3.7,3.7,3.7,3.7,3.7,3.7,3.7,3.7,1.9,1.9,1.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Consolidación

Apilamos los 40.336 archivos en una sola tabla en formato largo, añadiendo `pid` y `hosp`.

Dos decisiones de tipos de datos que vale la pena justificar:

- **`float32` en vez de `float64`** para las mediciones: los valores clínicos tienen a lo sumo
  dos decimales, así que `float64` no aporta precisión y duplica la memoria de una tabla de
  ~1,5 millones de filas.
- **`category` para `pid` y `hosp`**: son cadenas repetidas millones de veces; como categoría
  ocupan una fracción y las agrupaciones por paciente son mucho más rápidas.

In [5]:
df = io_datos.cargar_cohorte(cfg)
print(f"{len(df):,} filas-hora | {df.pid.nunique():,} pacientes | {df.shape[1]} columnas")
print(f"Memoria: {df.memory_usage(deep=True).sum() / 1024**2:.0f} MB")

Leyendo pacientes:   0%|          | 0/40336 [00:00<?, ?it/s]

Leyendo pacientes:   0%|          | 22/40336 [00:00<08:17, 80.97it/s]

Leyendo pacientes:   0%|          | 47/40336 [00:00<04:49, 139.09it/s]

Leyendo pacientes:   0%|          | 82/40336 [00:00<03:16, 204.77it/s]

Leyendo pacientes:   0%|          | 113/40336 [00:00<02:50, 236.54it/s]

...

Leyendo pacientes: 100%|█████████▉| 40269/40336 [02:48<00:00, 259.22it/s]

Leyendo pacientes: 100%|█████████▉| 40295/40336 [02:48<00:00, 254.77it/s]

Leyendo pacientes: 100%|█████████▉| 40321/40336 [02:48<00:00, 253.31it/s]

Leyendo pacientes: 100%|██████████| 40336/40336 [02:48<00:00, 239.04it/s]

1,552,210 filas-hora | 40,336 pacientes | 43 columnas
Memoria: 243 MB


## 4. Auditoría de integridad

Todo el proyecto se apoya en cuatro supuestos. Si alguno falla, se rompen cosas más adelante
de forma silenciosa, así que los verificamos ahora en vez de asumirlos.

### 4.1 ¿El reloj (`ICULOS`) es limpio?

Vamos a construir ventanas temporales (media de las últimas 6 h, pendiente de las últimas 24 h).
Eso presupone que las filas de un paciente están **ordenadas y son consecutivas hora a hora**.
Si hubiera saltos, una "ventana de 6 filas" no sería una ventana de 6 horas y las features
quedarían mal definidas.

In [6]:
reloj = df.groupby("pid", observed=True)["ICULOS"].agg(["min", "max", "count"])
contiguo = (reloj["max"] - reloj["min"] + 1 == reloj["count"])
empieza_en_1 = (reloj["min"] == 1)
ordenado = df.groupby("pid", observed=True)["ICULOS"].is_monotonic_increasing.all()

pd.Series({
    "pacientes con ICULOS contiguo (sin huecos)": f"{contiguo.mean() * 100:.2f} %",
    "pacientes que empiezan en ICULOS = 1": f"{empieza_en_1.mean() * 100:.2f} %",
    "orden correcto dentro de cada paciente": str(ordenado),
}).to_frame("resultado")

,resultado
pacientes con ICULOS contiguo (sin huecos),100.00 %
pacientes que empiezan en ICULOS = 1,78.20 %
orden correcto dentro de cada paciente,True


La buena noticia es que **no hay un solo hueco**: dentro de cada paciente las horas son
consecutivas, así que las ventanas rodantes son válidas.

Pero aparece algo que no esperábamos: **el 21,8 % de los pacientes no empieza en `ICULOS = 1`**.
Su registro arranca en la hora 2, 3, 5... Esto no es un hueco intermedio, es un arranque tardío,
y merece explicación antes de seguir. Veamos dónde se concentra:

In [7]:
resumen_inicio = reloj.assign(
    inicio_tardio=~empieza_en_1,
    hosp=df.groupby("pid", observed=True)["hosp"].first(),
    septico=df.groupby("pid", observed=True)["SepsisLabel"].max(),
)

resumen_inicio.groupby(["hosp", "inicio_tardio"], observed=True).agg(
    n_pacientes=("count", "size"),
    horas_mediana=("count", "median"),
    prev_sepsis_=("septico", lambda s: round(s.mean() * 100, 1)),
)

n_pacientes  horas_mediana  prev_sepsis_
hosp inicio_tardio                                          
A    False                12839           39.0           9.5
     True                  7497           37.0           7.7
B    False                18704           38.0           5.7
     True                  1296           37.0           6.4

El patrón es claro y es **una diferencia entre hospitales, no un error aleatorio**: el arranque
tardío afecta a ~37 % de los pacientes del hospital A y a solo ~6 % del B. La duración mediana
de la estancia es prácticamente la misma en ambos grupos, así que no se trata de pacientes
distintos, sino de **cómo cada hospital registró las primeras horas**.

Tres consecuencias prácticas que arrastramos al resto del proyecto:

1. **`ICULOS` sigue siendo un reloj válido**: mide horas reales desde el ingreso a UCI, y que
   falten las primeras filas no lo desplaza. No hay que reindexar nada.
2. **El "basal" del paciente hay que definirlo con cuidado**: en el notebook 05 calcularemos la
   desviación respecto a las primeras horas registradas. Para estos pacientes esas no son las
   primeras horas *reales* de UCI. Lo dejamos anotado como limitación explícita.
3. **Es la primera evidencia dura del *shift* entre hospitales.** Antes de mirar ni un signo
   vital ya sabemos que A y B no registran igual. Esto justifica por sí solo la estrategia de
   validación cruzada entre hospitales que aplicaremos en el notebook 06.

Guardamos el marcador para poder usarlo después como variable de control.

### 4.2 ¿La etiqueta es monótona?

`SepsisLabel` debería activarse una vez y quedarse en 1 hasta el alta: la sepsis no "se apaga"
dentro de la definición del challenge. Si encontráramos transiciones 1 → 0, el momento de
inicio del evento sería ambiguo y el utility score (que localiza el inicio con el primer 1)
daría resultados sin sentido.

In [8]:
# Buscamos cualquier transición de 1 a 0 dentro de un mismo paciente
cambios = df.groupby("pid", observed=True)["SepsisLabel"].diff()
n_apagados = int((cambios == -1).sum())
print(f"Transiciones 1 -> 0 encontradas: {n_apagados}")

Transiciones 1 -> 0 encontradas: 0


### 4.3 ¿Las variables demográficas son constantes por paciente?

`Age`, `Gender` y `HospAdmTime` describen al paciente, no a la hora. Si son constantes podemos
tratarlas aparte (sin ingeniería temporal) y usarlas para construir la tabla de pacientes del
dashboard. Conviene comprobarlo en vez de darlo por hecho.

In [9]:
demograficas = ["Age", "Gender", "HospAdmTime", "Unit1", "Unit2"]
variabilidad = df.groupby("pid", observed=True)[demograficas].nunique(dropna=True).max()
variabilidad.rename("máx. valores distintos dentro de un paciente").to_frame()

,máx. valores distintos dentro de un paciente
Age,1
Gender,1
HospAdmTime,1
Unit1,1
Unit2,1


### 4.4 ¿Hay valores clínicamente imposibles?

Antes de imputar cualquier cosa hay que saber si hay valores que directamente no pueden
existir en un ser humano. Un `O2Sat` de 120 % o una frecuencia cardiaca negativa serían errores
de registro, no variabilidad biológica, y arrastrarlos contaminaría medias y desviaciones.

Definimos rangos fisiológicos generosos a propósito: buscamos **imposibles**, no atípicos.
Un paciente de UCI con una frecuencia cardiaca de 180 está grave, pero es real y no se toca.

In [10]:
rangos_posibles = {
    "HR": (10, 300), "O2Sat": (0, 100), "Temp": (20, 45), "SBP": (20, 300),
    "MAP": (10, 250), "DBP": (10, 200), "Resp": (0, 80), "pH": (6.5, 8.0),
    "Glucose": (10, 1500), "Age": (0, 120),
}

fuera_de_rango = {}
for col, (lo, hi) in rangos_posibles.items():
    n = int(((df[col] < lo) | (df[col] > hi)).sum())
    if n:
        fuera_de_rango[col] = {"n_valores": n,
                               "porcentaje_": round(n / df[col].notna().sum() * 100, 4)}

pd.DataFrame(fuera_de_rango).T if fuera_de_rango else "Ningún valor fuera de los rangos fisiológicos posibles."

,n_valores,porcentaje_
Temp,2.0,0.0004
MAP,201.0,0.0148
DBP,103.0,0.0097
Resp,50.0,0.0038


Aparecen 356 valores sospechosos sobre 1,5 millones de filas. Son cantidades ínfimas, pero
antes de decidir qué hacer con ellos hay que **mirarlos**, porque "fuera de mi umbral
arbitrario" no es lo mismo que "imposible":

In [11]:
# ¿Qué valores concretos son? Un umbral redondo puede estar marcando valores extremos pero reales.
for col, (lo, hi) in rangos_posibles.items():
    extremos = df.loc[(df[col] < lo) | (df[col] > hi), col]
    if len(extremos):
        print(f"{col:10s} n={len(extremos):4d}  rango observado: {extremos.min():.1f} — {extremos.max():.1f}")

Temp       n=   2  rango observado: 50.0 — 50.0
MAP        n= 201  rango observado: 251.0 — 300.0
DBP        n= 103  rango observado: 201.0 — 300.0
Resp       n=  50  rango observado: 81.0 — 100.0


Ahora se puede decidir con criterio, y el criterio es distinto según el caso:

- **`Temp` = 50 °C** (2 registros): incompatible con la vida, sin ambigüedad posible. Es un
  error de digitación.
- **`MAP` y `DBP` que llegan exactamente hasta 300,0 mmHg**: el tope redondo e idéntico en ambas
  variables es la pista. No es casualidad fisiológica, es el límite de escala del monitor: el
  sensor satura y reporta su máximo. Además, una presión arterial media de 300 mmHg implicaría
  una sistólica aún mayor, algo incompatible con un paciente vivo. Artefacto típico de una línea
  arterial mal calibrada o purgada.
- **`Resp` entre 81 y 100** respiraciones por minuto: una taquipnea extrema real llega a ~60. Por
  encima de 80 lo que suele estar midiendo el monitor es movimiento del paciente, no ventilación.

**Decisión: convertir estos 356 valores a nulo, no eliminar las filas.** El razonamiento importa:
la fila contiene además frecuencia cardiaca, saturación y otras variables perfectamente válidas
de esa hora. Borrarla completa tiraría datos buenos por culpa de una sola celda mala. Al marcar
solo la celda como nula, el valor entra al mismo tratamiento de nulos que el resto y no
contamina medias ni desviaciones.

Es la única modificación de datos de este notebook, y es corrección de errores de registro,
no una decisión analítica.

In [12]:
n_antes = int(df[list(rangos_posibles)].notna().sum().sum())

for col, (lo, hi) in rangos_posibles.items():
    df.loc[(df[col] < lo) | (df[col] > hi), col] = pd.NA

n_despues = int(df[list(rangos_posibles)].notna().sum().sum())
print(f"Mediciones anuladas: {n_antes - n_despues} de {n_antes:,} "
      f"({(n_antes - n_despues) / n_antes * 100:.4f} % del total)")

Mediciones anuladas: 356 de 10,263,226 (0.0035 % del total)

## 5. Persistencia

Guardamos en **parquet** y no en CSV por dos razones concretas: el CSV perdería los tipos que
acabamos de fijar (todo volvería a `float64` y `object` al releer, deshaciendo el ahorro de
memoria) y pesaría varias veces más para 1,5 millones de filas.

Este archivo es el **punto de entrada único** de todos los notebooks siguientes: ninguno vuelve
a leer los `.psv`. Así garantizamos que todos trabajan exactamente sobre los mismos datos.

In [13]:
destino = io_datos.guardar_cohorte(df, cfg)
print(f"Guardado en {destino.relative_to(cfg['raiz'])} ({destino.stat().st_size / 1024**2:.0f} MB)")

Guardado en data\interim\cohorte_horaria.parquet (16 MB)


## 6. Qué sabemos al cerrar este notebook

**Estructura confirmada:** 40.336 pacientes repartidos casi por igual entre dos hospitales
(20.336 en A, 20.000 en B), 1.552.210 filas-hora, 40 variables clínicas más la etiqueta.

**Supuestos que se cumplen:** `ICULOS` es contiguo y está ordenado dentro de cada paciente — las
ventanas rodantes son válidas. `SepsisLabel` nunca pasa de 1 a 0, así que el inicio del evento
está bien definido y el utility score podrá localizarlo. `Age`, `Gender` y `HospAdmTime` son
constantes por paciente y pueden tratarse como atributos, no como series.

**Dos hallazgos que no esperábamos** y que cambian cosas más adelante:

1. **El 21,8 % de los pacientes no tiene registradas sus primeras horas de UCI**, y el fenómeno
   está muy concentrado en el hospital A (37 % vs 6 %). Es la primera evidencia de que los dos
   hospitales no registran igual, y obliga a matizar cómo definimos el "basal" de un paciente.
2. **356 mediciones fisiológicamente imposibles** (Temp de 50 °C, presiones y frecuencias
   respiratorias de artefacto). Se anularon las celdas, no las filas, para no descartar las
   mediciones válidas que las acompañan.

**El reto que ya se ve venir:** el registro es extremadamente disperso. Un paciente tiene sus
signos vitales casi cada hora, pero sus laboratorios aparecen unas pocas veces en toda la
estancia. Cuantificarlo bien y decidir qué hacer con ello es el trabajo del notebook 02, que es
donde de verdad se decide la calidad de este proyecto.

**Alcance de lo que se modificó aquí:** solo esas 356 celdas, que son errores de registro
inequívocos. Ninguna imputación, ninguna fila eliminada, ninguna variable transformada. Toda
decisión analítica queda para los notebooks siguientes, documentada donde se toma.
